<a href="https://colab.research.google.com/github/sundarbee/PythonWorkout/blob/main/MLOps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [159]:
# create a directory Structure
!mkdir -p MLChrun/data/raw
!mkdir -p MLChrun/data/processed
!mkdir -p MLChrun/src
#MLChrun/data/raw - Capture the raw data
#MLChrun/data/processed - store the clean data

#create a empty file
!touch MLChrun/src/ingest.py
!touch MLChrun/src/proprocess.py
!touch MLChrun/src/train.py

In [160]:
!ls -R MLChrun/

MLChrun/:
data  requirements.txt	src

MLChrun/data:
processed  raw

MLChrun/data/processed:
preprocessed_data.csv

MLChrun/data/raw:
data.csv

MLChrun/src:
ingest.py  proprocess.py  __pycache__  train.py

MLChrun/src/__pycache__:
ingest.cpython-312.pyc	proprocess.cpython-312.pyc  train.cpython-312.pyc


In [161]:
!touch MLChrun/requirements.txt

In [162]:
%%writefile /content/MLChrun/requirements.txt
mlflow
pyngrok

Overwriting /content/MLChrun/requirements.txt


In [163]:
!cat MLChrun/requirements.txt

mlflow
pyngrok


In [164]:
%%writefile /content/MLChrun/src/ingest.py

import pandas as pd
# to collect the data and store in our location further usage
def ingest_data():
  data_src = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
  data_dest= '/content/MLChrun/data/raw/data.csv'
  df = pd.read_csv(data_src)
  df.to_csv(data_dest, index=False)
  # return df
  print('Data Ingestoin Done in /content/MLChrun/data/raw/')
# ingest_data()
if __name__ == '__main__':
  ingest_data()



Overwriting /content/MLChrun/src/ingest.py


In [165]:
# !python /content/MLChrun/src/ingest.py.py


In [166]:
%%writefile /content/MLChrun/src/proprocess.py

import pandas as pd
def preprocess_data():
  # Set the src and dest path
  data_src = '/content/MLChrun/data/raw/data.csv'
  data_dest = '/content/MLChrun/data/processed'
  # load data from src
  df = pd.read_csv(data_src)
  print(df.shape)
  print(df.info())
  # ignore feature for preprocess
  ign_cols = ['customerID','Churn']
  # df = df.drop(ign_cols, axis=1)

  #  find cat and numeric colmns
  cat_cols = df.drop(ign_cols, axis=1).select_dtypes(include='object').columns
  print(cat_cols)
  num_cols = df.drop(ign_cols, axis=1).select_dtypes(exclude='object').columns
  print(num_cols)
  print(len(cat_cols),len(num_cols))

  # check null value and fix
  if df[cat_cols].isnull().sum().any():
    df[cat_cols].fillna(df[cat_cols].mode().iloc[0], inplace=True)

  if df[num_cols].isnull().sum().any():
    df[num_cols].fillna(df[num_cols].mean(), inplace=True)


  # fix the target value to numerical
  df['Churn'] = df['Churn'].map({'No':0, 'Yes':1}) # Corrected from 'Chrun' to 'Churn'
  print(df.info())

  # Do category encoding
  df_enc = pd.get_dummies( df[cat_cols], columns=cat_cols, drop_first=True,dtype=int)
  print(df_enc.shape)
  print(df_enc.head())

  df_new = pd.concat([df[num_cols], df_enc, df['Churn']], axis=1) # Include the target variable
  print(df['Churn'].unique())

  # store the data to processed folder
  df_new.to_csv(data_dest+'/preprocessed_data.csv', index=False)
  print('Data Preprocessing Done in /content/MLChrun/data/processed/')

  if __name__ == '__main__':
    preprocess_data()

# preprocess_data()

Overwriting /content/MLChrun/src/proprocess.py


In [167]:
!python /content/MLChrun/src/train.py

2026/02/01 05:31:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/01 05:31:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/01 05:31:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/01 05:31:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/01 05:31:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/01 05:31:58 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/01 05:31:58 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/01 05:31:58 INFO alembic.runtime.migration: Will assume non-transactional DDL.
Customer churn Classification
1
(100, 6560)
   SeniorCitizen  tenure  ...  TotalCharges_999.9  Churn
0              0       1  ...                   0      0

[1 rows x 6560 columns]
(80, 6559) (20, 6559) (80,) (20,)
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_lo

In [168]:
%%writefile /content/MLChrun/src/train.py

def train_model():
  import mlflow
  import sqlite3
  import pandas as pd
  from sklearn.model_selection import train_test_split
  from sklearn.linear_model import LogisticRegression
  from sklearn.metrics import accuracy_score, f1_score
  from sklearn.tree import DecisionTreeClassifier

  # store tracking info in a light weight db called sqlite
  MLFLOW_TRACKING_URI = 'sqlite:///myprojflow.db'
  mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
  experiment = mlflow.set_experiment('Customer churn Classification')
  print(experiment.name)

  #experiment = mlflow.set_experiment('Customer churn Classification')
  #print(experiment.name)
  print(experiment.experiment_id)

  data_src = '/content/MLChrun/data/processed/' # Corrected path to MLChrun/data/processed
  df = pd.read_csv(data_src+'preprocessed_data.csv',nrows=100)

  print(df.shape)
  print(df.head(1))
  #print(df.info())

  X = df.drop('Churn',axis=1)
  y = df['Churn']

  X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.2,random_state=42)
  print(X_train.shape,X_val.shape,y_train.shape,y_val.shape)

  run_name = 'LR L2'
  lm_name = 'lm '+run_name
  with mlflow.start_run(experiment_id = experiment.experiment_id,run_name=run_name ):
    model = LogisticRegression(penalty='l2', solver='lbfgs',max_iter=90)
    # Removed the conflicting line: model = LogisticRegression(penalty='l1',max_iter=50)

    model.fit(X_train,y_train)

    pred = model.predict(X_val)
    acc = accuracy_score(y_val,pred)
    f1 = f1_score(y_val,pred)

    #print(acc)
    #print(f1)
    mlflow.log_param('model',run_name)
    print(model)
    if isinstance(model, DecisionTreeClassifier):
      mlflow.log_param('max_depth',model.max_depth)
    elif isinstance(model, LogisticRegression):
      mlflow.log_param('penalty',model.penalty)
      mlflow.log_param('max_iter',model.max_iter)

    mlflow.log_metric('f1_score',f1)
    mlflow.log_metric('acc_score',acc)
    mlflow.sklearn.log_model(model,"model")
if __name__ == "__main__":
  train_model()


Overwriting /content/MLChrun/src/train.py


In [169]:
import mlflow
# store tracking info in a light weight db called sqlite
MLFLOW_TRACKING_URI = 'sqlite:///myprojflow.db'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment('Customer churn Classification')
print(experiment.name)


Customer churn Classification


In [170]:
!pip install -r /content/MLChrun/requirements.txt

In [171]:
import sys
sys.path.append('/content/MLChrun/src') # Corrected path to MLChrun/src
#from MLChrun.src.ingest import ingest_data
import ingest
import proprocess as preprocess # Corrected module name
import train

# to reload the model into current session memory
import importlib
importlib.reload(ingest)
importlib.reload(preprocess)
importlib.reload(train)

# calling/using the method from the module
ingest.ingest_data()
preprocess.preprocess_data()
train.train_model()

Data Ingestoin Done in /content/MLChrun/data/raw/
(7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-n

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/02/01 05:32:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogisticRegression(max_iter=90)


/usr/local/lib/python3.12/dist-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


In [172]:
!python /content/MLChrun/src/train.py

2026/02/01 05:33:02 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/01 05:33:02 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/01 05:33:02 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/01 05:33:02 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/01 05:33:02 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/01 05:33:02 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/01 05:33:03 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/01 05:33:03 INFO alembic.runtime.migration: Will assume non-transactional DDL.
Customer churn Classification
1
(100, 6560)
   SeniorCitizen  tenure  ...  TotalCharges_999.9  Churn
0              0       1  ...                   0      0

[1 rows x 6560 columns]
(80, 6559) (20, 6559) (80,) (20,)
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_lo

In [173]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('myprojflow.db')

e = pd.read_sql_query('SELECT * FROM experiments',conn)
display(e)

m = pd.read_sql_query('SELECT * FROM metrics',conn)
display(m)

p = pd.read_sql_query('SELECT * FROM params',conn)
display(p)

lm = pd.read_sql_query("SELECT * FROM logged_models;", conn)
display(lm)

summ = pd.read_sql_query('''
  SELECT e.name, m.key, MAX(m.value), p.key, p.value, r.name, r.run_uuid
  FROM experiments e
  JOIN runs r on e.experiment_id = r.experiment_id
  JOIN metrics m on r.run_uuid = m.run_uuid AND m.key='f1_score'
  JOIN params p on r.run_uuid = p.run_uuid
  --WHERE #e.experiment_id = m.experiment_id AND e.experiment_id = p.experiment_id
  ''',conn)
display(summ)

summ = pd.read_sql_query('''
  SELECT e.name, m.key, MAX(m.value), p.key, p.value, r.name, r.run_uuid
  FROM experiments e
  JOIN runs r on e.experiment_id = r.experiment_id
  JOIN metrics m on r.run_uuid = m.run_uuid AND m.key='acc_score'
  JOIN params p on r.run_uuid = p.run_uuid
  --WHERE #e.experiment_id = m.experiment_id AND e.experiment_id = p.experiment_id
  ''',conn)

display(summ)


,experiment_id,name,artifact_location,lifecycle_stage,creation_time,last_update_time
0,0,Default,/content/mlruns/0,active,1769916683811,1769916683811
1,1,Customer churn Classification,/content/mlruns/1,active,1769916683831,1769916683831


,key,value,timestamp,run_uuid,step,is_nan
0,f1_score,0.666667,1769916685315,ec40991f53104ae7b99ddd8f9b007136,0,0
1,acc_score,0.800000,1769916685338,ec40991f53104ae7b99ddd8f9b007136,0,0
2,f1_score,0.666667,1769917245534,8e79a8eb4b9a4132bca13ddb669ed588,0,0
3,acc_score,0.800000,1769917245546,8e79a8eb4b9a4132bca13ddb669ed588,0,0
4,f1_score,0.666667,1769920278873,fde63bd16977409fbadd2d7c79d49c42,0,0
5,acc_score,0.800000,1769920278889,fde63bd16977409fbadd2d7c79d49c42,0,0
6,f1_score,0.666667,1769923920808,01965b1c3bf4401b97de3121562b259c,0,0
7,acc_score,0.800000,1769923920823,01965b1c3bf4401b97de3121562b259c,0,0
8,f1_score,0.666667,1769923970285,1a73936017ae4284ac6b384ca2f85ad6,0,0
9,acc_score,0.800000,1769923970299,1a73936017ae4284ac6b384ca2f85ad6,0,0


,key,value,run_uuid
0,model,LR L2,ec40991f53104ae7b99ddd8f9b007136
1,penalty,l2,ec40991f53104ae7b99ddd8f9b007136
2,max_iter,90,ec40991f53104ae7b99ddd8f9b007136
3,model,LR L2,8e79a8eb4b9a4132bca13ddb669ed588
4,penalty,l2,8e79a8eb4b9a4132bca13ddb669ed588
5,max_iter,90,8e79a8eb4b9a4132bca13ddb669ed588
6,model,LR L2,fde63bd16977409fbadd2d7c79d49c42
7,penalty,l2,fde63bd16977409fbadd2d7c79d49c42
8,max_iter,90,fde63bd16977409fbadd2d7c79d49c42
9,model,LR L2,01965b1c3bf4401b97de3121562b259c


,model_id,experiment_id,name,artifact_location,creation_timestamp_ms,last_updated_timestamp_ms,status,lifecycle_stage,model_type,source_run_id,status_message
0,m-7de9a7a088e94edb926674fc512663e0,1,model,/content/mlruns/1/models/m-7de9a7a088e94edb926...,1769916685375,1769916691754,2,active,None,ec40991f53104ae7b99ddd8f9b007136,None
1,m-96ca4eee59e04c7ab761d15c800c8995,1,model,/content/mlruns/1/models/m-96ca4eee59e04c7ab76...,1769917245580,1769917251477,2,active,None,8e79a8eb4b9a4132bca13ddb669ed588,None
2,m-76ac42ca9de74d3d83ed97aafb5a908f,1,model,/content/mlruns/1/models/m-76ac42ca9de74d3d83e...,1769920278925,1769920284878,2,active,None,fde63bd16977409fbadd2d7c79d49c42,None
3,m-c028352a9eb34fb09147d2e7994164a6,1,model,/content/mlruns/1/models/m-c028352a9eb34fb0914...,1769923920856,1769923927207,2,active,None,01965b1c3bf4401b97de3121562b259c,None
4,m-1cd57f83456d4f5c98b2b4412e655e1c,1,model,/content/mlruns/1/models/m-1cd57f83456d4f5c98b...,1769923970329,1769923975424,2,active,None,1a73936017ae4284ac6b384ca2f85ad6,None
5,m-6d264c1419e248dfb36ffb1232af619f,1,model,/content/mlruns/1/models/m-6d264c1419e248dfb36...,1769923987000,1769923987000,1,active,None,1461829b953247dca868d77d3d2d5054,None


,name,key,MAX(m.value),key,value,name,run_uuid
0,Customer churn Classification,f1_score,0.666667,model,LR L2,LR L2,ec40991f53104ae7b99ddd8f9b007136


,name,key,MAX(m.value),key,value,name,run_uuid
0,Customer churn Classification,acc_score,0.8,model,LR L2,LR L2,ec40991f53104ae7b99ddd8f9b007136


In [174]:
# to pass authtoken access  and start ngrok tunnelling

import getpass

ngrok_token = getpass.getpass("Please enter your ngrok token: ")
get_ipython().system_raw(f'ngrok authtoken {ngrok_token}')
print('Ngrok has been setup')


Please enter your ngrok token: ··········
Ngrok has been setup


In [175]:
get_ipython().system_raw('mlflow ui')


KeyboardInterrupt



In [184]:
get_ipython().system_raw('mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 0.0.0.0 --port 5000 --allowed-hosts "*" &')
get_ipython().system_raw('ngrok http 5000 --log stdout &')

In [177]:
# !pkill -f ngrok
# !pkill -f mlflow

In [178]:
!lsof -i :5000

In [185]:
!curl -s http://localhost:4040/api/tunnels

{"tunnels":[{"name":"command_line","ID":"c986a565c61485d42fa7541354dacbb3","uri":"/api/tunnels/command_line","public_url":"https://disappointed-robeless-katheleen.ngrok-free.dev","proto":"https","config":{"addr":"http://localhost:5000","inspect":true},"metrics":{"conns":{"count":0,"gauge":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0},"http":{"count":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0}}}],"uri":"/api/tunnels"}


In [180]:
from mlflow import search_runs
runs = mlflow.search_runs(order_by=['metrics.acc_score DESC'])
best_run = runs.iloc[0]
print('best acc-score',best_run['metrics.acc_score'])
print('best f1-score',best_run['metrics.f1_score'])
print(best_run)

best acc-score 0.8
best f1-score 0.6666666666666666
run_id                                      1461829b953247dca868d77d3d2d5054
experiment_id                                                              1
status                                                                FAILED
artifact_uri               /content/mlruns/1/1461829b953247dca868d77d3d2d...
start_time                                  2026-02-01 05:33:04.650000+00:00
end_time                                    2026-02-01 05:33:10.283000+00:00
metrics.f1_score                                                    0.666667
metrics.acc_score                                                        0.8
params.max_iter                                                           90
params.model                                                           LR L2
params.penalty                                                            l2
tags.mlflow.source.name                        /content/MLChrun/src/train.py
tags.mlflow.source.type 

In [181]:
mlflow.register_model(
    f'runs:/{best_run["run_id"]}/model','BestAccModel'
)

Registered model 'BestAccModel' already exists. Creating a new version of this model...
2026/02/01 05:33:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 1461829b953247dca868d77d3d2d5054 has no artifacts at artifact path 'model', registering model based on models:/m-6d264c1419e248dfb36ffb1232af619f instead
Created version '2' of model 'BestAccModel'.


<ModelVersion: aliases=[], creation_timestamp=1769924004894, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1769924004894, metrics=None, model_id=None, name='BestAccModel', params=None, run_id='1461829b953247dca868d77d3d2d5054', run_link=None, source='models:/m-6d264c1419e248dfb36ffb1232af619f', status='READY', status_message=None, tags={}, user_id=None, version=2>

In [182]:
from mlflow.entities.model_registry import model_version
import pandas as pd
import mlflow.pyfunc
# do prediction from the best model
model_name = "BestAccModel"
model_version=1
model = mlflow.pyfunc.load_model(
    model_uri=f"models:/{model_name}/{model_version}"
)

# READ test data
df_test = pd.read_csv("/content/MLChrun/data/processed/preprocessed_data.csv",header=0)
print(df_test.shape)
display(df_test.tail(1))
df_test = df_test.tail(1)

testX = df_test.drop('Churn',axis=1)
testY = df_test['Churn']

predictions = model.predict(testX)
print(predictions,testY.values)



(7043, 6560)


,SeniorCitizen,tenure,MonthlyCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,TotalCharges_996.45,TotalCharges_996.85,TotalCharges_996.95,TotalCharges_997.65,TotalCharges_997.75,TotalCharges_998.1,TotalCharges_999.45,TotalCharges_999.8,TotalCharges_999.9,Churn
7042,0,66,105.65,1,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


[0] [0]


In [203]:
#move the selected model to productionanize stage
from mlflow.tracking import MlflowClient
client = MlflowClient()
model_name = "BestAccModel"
model_version = 1
model_stage = "Production"

client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=model_stage,
    archive_existing_versions=False
)


/tmp/ipython-input-3563900281.py:8: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1769921532387, current_stage='Production', deployment_job_state=None, description=None, last_updated_timestamp=1769925117227, metrics=None, model_id=None, name='BestAccModel', params=None, run_id='fde63bd16977409fbadd2d7c79d49c42', run_link=None, source='models:/m-76ac42ca9de74d3d83ed97aafb5a908f', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [204]:
get_ipython().system_raw('mlflow models serve -m "models:/BestAccModel/Production" -h 0.0.0.0 -p 5001 --no-conda &')

In [210]:
!lsof -i :5001

COMMAND   PID USER   FD   TYPE  DEVICE SIZE/OFF NODE NAME
uvicorn 37574 root   15u  IPv4 1153694      0t0  TCP *:5001 (LISTEN)


In [209]:
!lsof -i :5000

COMMAND   PID USER   FD   TYPE  DEVICE SIZE/OFF NODE NAME
python3 36642 root    3u  IPv4 1128652      0t0  TCP *:5000 (LISTEN)
python3 36649 root    3u  IPv4 1128652      0t0  TCP *:5000 (LISTEN)
python3 36650 root    3u  IPv4 1128652      0t0  TCP *:5000 (LISTEN)
python3 36651 root    3u  IPv4 1128652      0t0  TCP *:5000 (LISTEN)
python3 36652 root    3u  IPv4 1128652      0t0  TCP *:5000 (LISTEN)


In [211]:
from pyngrok import ngrok
import json
import requests

# Ensure all ngrok tunnels are closed before trying to open a new one
# This is a workaround for the 5-tunnel limit on free ngrok accounts.
ngrok.kill()

server_url = ngrok.connect(5001)
print(server_url)
url = server_url.public_url + '/invocations'
print(url)

#print(testX.shape)
#print(testX.head(1))
mydata = {"instances" : testX.values.tolist()}
print(mydata)

response = requests.post(url, json=mydata)
print(response.json())

NgrokTunnel: "https://disappointed-robeless-katheleen.ngrok-free.dev" -> "http://localhost:5001"
https://disappointed-robeless-katheleen.ngrok-free.dev/invocations
{'instances': [[0.0, 66.0, 105.65, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0

In [222]:
url = 'https://disappointed-robeless-katheleen.ngrok-free.dev/invocations'
mydata = {"instances" : testX.values.tolist()}
print(mydata)

response = requests.post(url, json=mydata)
print(response.json())

{'instances': [[0.0, 66.0, 105.65, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

In [221]:
import json
import requests
from pyngrok import ngrok

# Ensure all ngrok tunnels are closed before trying to open a new one
# This is a workaround for the 5-tunnel limit on free ngrok accounts.
ngrok.kill()
server_url = ngrok.connect(5001)
public_url = server_url.public_url

mlflow_predict_url = f"{public_url}/invocations"

# Assuming testX is available from previous execution
mydata_json = json.dumps({"instances" : testX.values.tolist()})

# Construct the curl command
curl_command = f"curl -X POST -H 'Content-Type: application/json' -d '{mydata_json}' {mlflow_predict_url}"

# Execute the curl command
print(f"Executing: {curl_command}")
!{curl_command}

Executing: curl -X POST -H 'Content-Type: application/json' -d '{"instances": [[0.0, 66.0, 105.65, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0